![Esquema]( /Workspace/Shared/schema.png )


In [0]:
%py
'''
PROJETO BIKESTORE - PIPELINE DE DADOS 

1. ARQUITETURA DE ARMAZENAMENTO
   - Catalog: bikesales (Unity Catalog)
   - Storage Account: stgdatalakebr01
   
   Estrutura no Storage Account:
   ├── raw/                       # Container para dados crus (CSV originais)
   │   └── *.csv                  # Arquivos fonte (brands, customers, orders, etc.)
   │
   ├── bronze/                    # Container para dados em Delta (reservado para External Location)
   │   └── [tabelas]/             # Dados processados em formato Delta
   │
   └── (silver/gold)              # Para futuras implementações

2. ESTRUTURA NO UNITY CATALOG (DESCOBERTAS E LIMITAÇÕES)
   
   ✅ O QUE FUNCIONOU:
   ├── Catalog: bikesales
   │   ├── Schema: raw
   │   │   └── Volume EXTERNAL: files
   │   │       └── LOCATION: 'abfss://raw@stgdatalakebr01.dfs.core.windows.net/'
   │   │           ├── brands.csv
   │   │           ├── customers.csv
   │   │           ├── orders.csv
   │   │           └── ... (demais arquivos)
   │   │
   │   ├── Schema: bronze (dados crus em Delta)
   │   │   └── ⚠️ Não é possível criar tabelas em Volumes!
   │   │       ❌ CREATE TABLE com LOCATION em Volume → ERRO
   │   │       ✅ Solução: Managed Tables (sem LOCATION)
   │   │
   │   └── Schema: views (RECOMENDADO - SOLUÇÃO MAIS SIMPLES)
   │       └── Views persistentes sobre os dados no Volume
   │           ├── vw_brands
   │           ├── vw_customers
   │           ├── vw_orders
   │           └── ... (demais views)


 3. LIÇÕES APRENDIDAS
   
   📌 Volumes são para arquivos não-tabelares (CSV, JSON, PDF, etc.)
   📌 Não se pode criar tabelas Delta com LOCATION apontando para Volumes
   📌 Views sobre Volumes funcionam perfeitamente
   📌 Para tabelas, usar Managed Tables ou External Locations (fora dos Volumes)
   📌 A abordagem mais simples e funcional: USAR VIEWS


'''

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS bikesales.raw
COMMENT 'Schema para dados crus (raw) do projeto BikeStore';

In [0]:
%sql
CREATE EXTERNAL VOLUME IF NOT EXISTS bikesales.raw.files
COMMENT 'Volume para arquivos CSV no container raw'
LOCATION 'abfss://raw@stgdatalakebr01.dfs.core.windows.net/';

In [0]:
%sql
SHOW VOLUMES IN bikesales.raw;

In [0]:
%python
display(dbutils.fs.ls('/Volumes/bikesales/raw/files/'))

In [0]:
# Seus caminhos
volume_path = '/Volumes/bikesales/raw/files/'
bronze_path = '/Volumes/bikesales/raw/bronze/'  # ← usando Volume para escrita

In [0]:
%sql
CREATE EXTERNAL VOLUME IF NOT EXISTS bikesales.raw.bronze
COMMENT 'Volume para dados na camada bronze'
LOCATION 'abfss://bronze@stgdatalakebr01.dfs.core.windows.net/'

In [0]:
from datetime import datetime
data_processamento = datetime.now().strftime('%Y-%m-%d')
print(f"Data de processamento: {data_processamento}")

In [0]:
%python
from pyspark.sql.functions import lit, current_date
from datetime import datetime


for arquivo in dbutils.fs.ls('/Volumes/bikesales/raw/files/'):
    if arquivo.path.endswith('.csv'):
        nome = arquivo.name.replace('.csv', '')
        destino = f"{bronze_path}{nome}"
        
        # Apagar pasta anterior se existir
        try:
            dbutils.fs.rm(destino, recurse=True)
            print(f"🗑️ Pasta antiga removida: {nome}")
        except:
            pass
        
        print(f"Processando: {nome}")
        
        df = spark.read.option("header", "true") \
                       .option("inferSchema", "true") \
                       .csv(arquivo.path)
        
        df = df.withColumn("data_processamento", lit(data_processamento)) \
               .withColumn("data_carga", current_date())
        
        df.write.format("delta") \
          .mode("overwrite") \
          .option("overwriteSchema", "true") \
          .save(destino)
        
        print(f" {nome}: {df.count()} linhas, {len(df.columns)} colunas")

print("Todos processados!")

In [0]:
%python

df_stores = spark.read.format("delta") \
    .load("/Volumes/bikesales/raw/bronze/stores")

display(df_stores)

In [0]:
%sql
SELECT * FROM delta.`/Volumes/bikesales/raw/bronze/brands`;

![image_1777408986771.png](./image_1777408986771.png "image_1777408986771.png")

In [0]:
%sql
USE CATALOG bikesales; 

SHOW VIEWS IN views;

In [0]:
%sql
CREATE OR REPLACE VIEW bikesales.views.vw_receita
AS
SELECT 
    s.store_name,
    COUNT(DISTINCT o.order_id) as total_pedidos,
    ROUND(SUM(oi.quantity * oi.list_price * (1 - oi.discount)), 2) as receita
FROM bikesales.views.vw_orders o
JOIN bikesales.views.vw_stores s ON o.store_id = s.store_id
JOIN bikesales.views.vw_order_items oi ON o.order_id = oi.order_id
GROUP BY s.store_name;


![image_1777411243370.png](./image_1777411243370.png "image_1777411243370.png")